In [4]:
# Install PyBaMM - run once
%pip install pybamm

Note: you may need to restart the kernel to use updated packages.


In [5]:
#Import libraries
import pybamm
from matplotlib import pyplot
import pandas as pd
import numpy as np
import random
import os

In [6]:
model = pybamm.lithium_ion.DFN()
print("Available variables:")
for var in sorted(model.variables.keys()):
    if 'state' in var.lower() or 'charge' in var.lower() or 'soc' in var.lower():
        print(var)

Available variables:
Discharge capacity [A.h]
Discharge energy [W.h]


Power vs time for 4 Duty cycles from ANL -  https://www.anl.gov/taps/d3-2020-tesla-model-3

1. Urban (EPA UDDS)
2. Gentle highway (EPA HwFET)
3. Agressive highway (EPA US06)
4. WLTP (UNECE World Harmonized)
The battery power pack values were scaled to cell level by dividing power by 4416 (# of cells in the pack)
Condensed files saved in InputData\72F --> 72F is controlled test cell temperature

+ One more cycle to get rich dataset
5. Pseudo random power draw

Vehicle has 96s46p configuration. 96 cell in series and 46 in parallel
    -- Voltage get added in series
    -- Charge gets added in parallel

To generate highfidelity synthetic data for current, voltage and state of charge (SOC)
    -- We will demand the above power profiles from the high-fidelty DFN model -- Chen2020
    -- Note each power profile will only deplete the cell partially, so we will run it back to back until the cell is empty 

In [7]:
# Load drive cycle data
base_path = os.path.join("InputData", "72F")

# Dictionary mapping standard names to your exact file names
file_map = {
    "HWFET": "ANL_62005016_3_GentleHighway_HwFET.xlsx",
    "UDDS": "ANL_62005016_45_Urban_UDDS.xlsx",
    "US06": "ANL_62005016_67_AggressiveHighway_US06.xlsx",
    "WLTP": "ANL_62006002_1234_UNECE_2xWLTP.xlsx"
}

cycle_data_arrays = {}
all_power_values = []

for name, filename in file_map.items():
    filepath = os.path.join(base_path, filename)
    
    # Read the Excel file
    try:
        df = pd.read_excel(filepath)
        
        # Extract Time and Cell Power
        time_s = df["Time_s"].to_numpy()
        power_cell_w = df["Power_Cell_W"].to_numpy()
        
        # PyBaMM requires a 2-column array: [Time, Data]
        cycle_array = np.column_stack((time_s, power_cell_w))
        cycle_data_arrays[name] = cycle_array
        
        # Store power values to determine global limits for the random cycle
        all_power_values.extend(power_cell_w)
        print(f"  -> Successfully loaded {name}")
        
    except FileNotFoundError:
        print(f"  -> ERROR: Could not find {filepath}")
    except KeyError as e:
        print(f"  -> ERROR: Missing column {e} in {filename}")


  -> Successfully loaded HWFET
  -> Successfully loaded UDDS
  -> Successfully loaded US06
  -> Successfully loaded WLTP


In [8]:
# Build pseudo random pulses
max_discharge_w = max(all_power_values)
max_charge_w = min(all_power_values) # This will be negative (Regen)

print(f"\nExtracted Power Limits: {max_charge_w:.2f} W (Charge) to {max_discharge_w:.2f} W (Discharge)")


# Build Experiments
experiments_dict = {}

# A. Standard Drive Cycles (Repeated to v_min)
for name, data_array in cycle_data_arrays.items():
    experiments_dict[name] = pybamm.Experiment(
        [pybamm.step.power(data_array)] * 100, # Repeat the trace up to 100 times
        termination="2.5 V",                   # Stop exactly when v_min is reached
        period="0.1 second"
    )

# B. Pseudo-Random Cycle
random.seed(42) # Ensure reproducibility 
pulse_steps = []

for _ in range(50): # Generate a block of 50 distinct random pulses
    # Pick a random power between the max charge and max discharge limits
    power = round(random.uniform(max_charge_w, max_discharge_w), 2)
    duration = random.randint(5, 45)
    rest = random.randint(10, 60)
    
    # PyBaMM's string parser expects positive numbers and explicit keywords
    if power > 0:
        pulse_steps.append(f"Discharge at {power} W for {duration} seconds")
    elif power < 0:
        pulse_steps.append(f"Charge at {abs(power)} W for {duration} seconds")
    
    pulse_steps.append(f"Rest for {rest} seconds")

experiments_dict["Pseudo-Random Limit Based"] = pybamm.Experiment(
    pulse_steps * 100,     # Repeat the 50-pulse block up to 100 times to guarantee depletion
    termination="2.5 V",    # Stop exactly when v_min is reached
    period="0.1 second"
)


Extracted Power Limits: -22.68 W (Charge) to 27.22 W (Discharge)


In [9]:
#  Initialize PyBAMM

model = pybamm.lithium_ion.DFN()
parameter_values = pybamm.ParameterValues("Chen2020") #NMC811 21700 cell ~ 2020 Tesla Model 3 Long Range


solutions = {}

for name, exp in experiments_dict.items():
    print(f"\n--- Simulating: {name} ---")
    sim = pybamm.Simulation(
        model, 
        parameter_values=parameter_values, 
        experiment=exp
    )
    # The Casadi solver handles the DFN model gradients automatically
    solutions[name] = sim.solve()
    print(f"--> {name} complete!")


--- Simulating: HWFET ---


2026-04-23 20:55:12.700 - [WARNING] callbacks.on_experiment_infeasible_event(254): 

	Experiment is infeasible: 'event: Minimum voltage [V]' was triggered during 'Step([[0.00000000e+00 5.93297101e-02]
 [1.00000000e-01 6.02355072e-02]
 [2.00000000e-01 6.02355072e-02]
 ...
 [7.63600000e+02 6.72554348e-02]
 [7.63700000e+02 6.65760870e-02]
 [7.63800000e+02 6.61231884e-02]], duration=763.8)'. The returned solution only contains up to step 1 of cycle 43. 


--> HWFET complete!

--- Simulating: UDDS ---


2026-04-23 21:22:04.830 - [WARNING] callbacks.on_experiment_infeasible_event(254): 

	Experiment is infeasible: 'event: Minimum voltage [V]' was triggered during 'Step([[0.00000000e+00 2.22146739e-01]
 [1.00000000e-01 2.21920290e-01]
 [2.00000000e-01 2.20788043e-01]
 ...
 [1.37350000e+03 2.22599638e-01]
 [1.37360000e+03 2.21240942e-01]
 [1.37370000e+03 2.19202899e-01]], duration=1373.7)'. The returned solution only contains up to step 1 of cycle 24. 


--> UDDS complete!

--- Simulating: US06 ---


2026-04-23 21:29:37.569 - [WARNING] callbacks.on_experiment_infeasible_event(254): 

	Experiment is infeasible: 'event: Minimum voltage [V]' was triggered during 'Step([[0.00000000e+00 7.15579710e-02]
 [1.00000000e-01 7.13315217e-02]
 [2.00000000e-01 7.13315217e-02]
 ...
 [5.99400000e+02 7.42753623e-02]
 [5.99500000e+02 7.40489130e-02]
 [5.99600000e+02 7.42753623e-02]], duration=599.6)'. The returned solution only contains up to step 1 of cycle 40. 


--> US06 complete!

--- Simulating: WLTP ---


2026-04-23 21:46:55.374 - [WARNING] callbacks.on_experiment_infeasible_event(254): 

	Experiment is infeasible: 'event: Minimum voltage [V]' was triggered during 'Step([[0.00000000e+00 7.26902174e-02]
 [1.00000000e-01 7.26902174e-02]
 [2.00000000e-01 7.38224638e-02]
 ...
 [3.65480000e+03 2.51358696e-02]
 [3.65490000e+03 2.49094203e-02]
 [3.65500000e+03 2.49094203e-02]], duration=3655.00000000311)'. The returned solution only contains up to step 1 of cycle 15. 


--> WLTP complete!

--- Simulating: Pseudo-Random Limit Based ---
--> Pseudo-Random Limit Based complete!


In [10]:
# Plotting
if solutions:
    print("\nGenerating plots...")
    for name, sol in solutions.items():
        plot = pybamm.QuickPlot(
            sol,
            output_variables=[
                "Terminal power [W]", 
                "Current [A]",
                "Terminal voltage [V]",
                "Discharge capacity [A.h]",
                "X-averaged cell temperature [K]"
            ],
            labels=[name] 
        )
        plot.dynamic_plot()


Generating plots...


interactive(children=(FloatSlider(value=0.0, description='t', max=9.109443853051562, step=0.09109443853051563)…

interactive(children=(FloatSlider(value=0.0, description='t', max=8.960019257180827, step=0.08960019257180826)…

interactive(children=(FloatSlider(value=0.0, description='t', max=6.655824021736152, step=0.06655824021736152)…

interactive(children=(FloatSlider(value=0.0, description='t', max=14.6523312027294, step=0.146523312027294), O…

interactive(children=(FloatSlider(value=0.0, description='t', max=17.884444444444444, step=0.17884444444444444…

In [ ]:
output_dir = "DFN_Data"

# Create the directory if it doesn't exist
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"\nCreated directory: {output_dir}")

print(f"\nSaving simulation data to '{output_dir}' at exactly 0.1s intervals...")

# Get parameter values for SOC calculation
parameter_values = pybamm.ParameterValues("Chen2020")
nominal_capacity = parameter_values["Nominal cell capacity [A.h]"]

# Define exactly which variables you want exported to the CSV
variables_to_export = [
    "Time [s]",
    "Terminal power [W]",
    "Current [A]",
    "Terminal voltage [V]",
    "Discharge capacity [A.h]",
    "State of Charge"
]

for name, sol in solutions.items():
    # Format the dictionary key into a safe filename
    safe_name = name.replace(" ", "_")
    filepath = os.path.join(output_dir, f"{safe_name}_results.csv")
     
    try:
        # 1. Create a uniform time array with exactly 0.1s steps
        t_max = sol["Time [s]"].entries[-1]
        t_export = np.arange(0, t_max, 0.1)
        
        # 2. Extract and interpolate the data for each variable
        export_dict = {}
        for var in variables_to_export:
            if var == "Time [s]":
                export_dict[var] = t_export
            elif var == "State of Charge":
                export_dict[var] = 1 - (sol["Discharge capacity [A.h]"](t_export) / nominal_capacity)
            else:
                # Passing the time array into the solution variable automatically interpolates it!
                export_dict[var] = sol[var](t_export) 
                
        # 3. Convert to a Pandas DataFrame and save to CSV
        df_export = pd.DataFrame(export_dict)
        df_export.to_csv(filepath, index=False)
        print(f"  -> Saved (10 Hz resolution): {filepath}")
        
    except Exception as e:
        print(f"  -> ERROR saving {name}: {e}")

print("\nAll data successfully exported!")


Saving simulation data to 'DFN_Data' at exactly 0.1s intervals...
  -> Saved (10 Hz resolution): DFN_Data\HWFET_results.csv
  -> Saved (10 Hz resolution): DFN_Data\UDDS_results.csv
  -> Saved (10 Hz resolution): DFN_Data\US06_results.csv
  -> Saved (10 Hz resolution): DFN_Data\WLTP_results.csv
  -> Saved (10 Hz resolution): DFN_Data\Pseudo-Random_Limit_Based_results.csv

All data successfully exported!
